In [41]:
import multilayerGM as gm

n_nodes = 50
n_layers = 20
p = 0.8
theta = 0.2
n_sets = 5
dt = gm.dependency_tensors.UniformMultiplex(n_nodes, n_layers, p)
null = gm.DirichletNull(layers=dt.shape[1:], theta=theta, n_sets=n_sets)

partition = gm.sample_partition(dependency_tensor=dt, null_distribution=null)
multinet = gm.multilayer_DCSBM_network(partition, mu=0.0, k_min=2, k_max=10, t_k=-2)

/Users/acw721/Desktop/research/linkstream/.venv/lib/python3.12/site-packages/multilayerGM/networks.py:93: RuntimeWarning: invalid value encountered in divide
  sigma = [degrees[g] / sum(degrees[g]) for g in partition_map]


In [42]:
print(set(partition.values()))

{0, 1, 2, 3, 4}


In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from collections import defaultdict


def generate_temporal_network_from_dict(
    label_dict,
    output_csv="synthetic_temporal_network.csv",
    link_node_ratio=4.0,
    mu=0.2,
    poisson_rate=1.0,
    seed=42,
    undirected=True,
):
    """
    根据 {(u, layer): label} 生成时序交互网络 CSV

    参数
    ----
    label_dict : dict
        形如 {(u, layer): label}
        例如:
        {
            (0, 0): 1,
            (1, 0): 1,
            (2, 0): 2,
            (0, 1): 1,
            (1, 1): 2,
            ...
        }

    output_csv : str
        输出 CSV 路径

    link_node_ratio : float
        控制每层总事件数的强度。
        第 l 层若有 n_l 个节点，则
            M_l ~ Poisson(link_node_ratio * n_l / 2)

    mu : float in [0, 1]
        控制社区间事件比例
            mu = 0   -> 全是社区内事件
            mu = 1   -> 全是社区间事件

    poisson_rate : float > 0
        泊松过程到达率 lambda。
        时间间隔 Delta t ~ Exponential(poisson_rate)

    seed : int
        随机种子

    undirected : bool
        是否按无向图处理。
        目前输出时 source < destination（如果节点可比较）

    返回
    ----
    df : pd.DataFrame
        包含列:
        source, destination, timestamp, source_commu, destination_commu
    """
    if not (0.0 <= mu <= 1.0):
        raise ValueError("mu must be in [0, 1].")
    if link_node_ratio < 0:
        raise ValueError("link_node_ratio must be >= 0.")
    if poisson_rate <= 0:
        raise ValueError("poisson_rate must be > 0.")

    rng = np.random.default_rng(seed)

    # -------- 1) 整理成 layer -> {u: label} --------
    layer_to_nodes = defaultdict(dict)
    for (u, layer), label in label_dict.items():
        layer_to_nodes[layer][u] = label

    all_layers = sorted(layer_to_nodes.keys())

    records = []
    current_time = 0.0  # 全局时间，按 layer 顺序往后累加

    for layer in all_layers:
        node_label_map = layer_to_nodes[layer]
        nodes = list(node_label_map.keys())
        n = len(nodes)

        if n < 2:
            continue

        # -------- 2) 该层总事件数 --------
        expected_events = link_node_ratio * n / 2.0
        m_total = rng.poisson(expected_events)

        if m_total <= 0:
            continue

        # -------- 3) 列出候选节点对：社区内 / 社区间 --------
        in_pairs = []
        out_pairs = []

        for u, v in combinations(nodes, 2):
            lu = node_label_map[u]
            lv = node_label_map[v]
            if lu == lv:
                in_pairs.append((u, v))
            else:
                out_pairs.append((u, v))

        # 如果某一类为空，要自动把事件都给另一类
        if len(in_pairs) == 0 and len(out_pairs) == 0:
            continue
        elif len(in_pairs) == 0:
            m_in = 0
            m_out = m_total
        elif len(out_pairs) == 0:
            m_out = 0
            m_in = m_total
        else:
            m_out = rng.binomial(m_total, mu)
            m_in = m_total - m_out

        # -------- 4) 先采样节点对 --------
        sampled_pairs = []

        if m_in > 0:
            idx_in = rng.integers(low=0, high=len(in_pairs), size=m_in)
            sampled_pairs.extend([in_pairs[i] for i in idx_in])

        if m_out > 0:
            idx_out = rng.integers(low=0, high=len(out_pairs), size=m_out)
            sampled_pairs.extend([out_pairs[i] for i in idx_out])

        # -------- 5) 用泊松过程生成这一层事件时间 --------
        # inter-arrival ~ Exponential(rate = poisson_rate)
        inter_arrivals = rng.exponential(scale=1.0 / poisson_rate, size=len(sampled_pairs))
        event_times = current_time + np.cumsum(inter_arrivals)

        # 当前层结束后，把 current_time 推到最后一个事件时间
        current_time = float(event_times[-1])

        # -------- 6) 打乱 pair 和 time 的对应关系（可选）--------
        # 让“先采到的 pair”不一定对应“最早发生的时间”
        perm = rng.permutation(len(sampled_pairs))
        sampled_pairs = [sampled_pairs[i] for i in perm]

        # event_times 保持递增，这样最终整个数据按时间是自然有序的
        for (u, v), t in zip(sampled_pairs, event_times):
            if undirected:
                try:
                    if u > v:
                        u, v = v, u
                except TypeError:
                    # 如果节点 ID 类型不可比较，就保持原样
                    pass

            records.append({
                "source": u,
                "destination": v,
                "timestamp": int(t),
                "source_commu": node_label_map[u],
                "destination_commu": node_label_map[v],
            })

    df = pd.DataFrame(records)

    if not df.empty:
        df = df.sort_values("timestamp").reset_index(drop=True)

    df.to_csv(output_csv, index=False)
    return df


# =========================
# 使用示例
# =========================
if __name__ == "__main__":

    df = generate_temporal_network_from_dict(
        label_dict=partition,
        output_csv="synthetic_temporal_network.csv",
        link_node_ratio=5.0,
        mu=0.2,
        poisson_rate=0.5,
        seed=42,
    )

    print(df.head(20))
    print(f"\nSaved to synthetic_temporal_network.csv, total events = {len(df)}")

    source  destination  timestamp  source_commu  destination_commu
0        9           22          3             2                  1
1       13           19          7             1                  1
2       28           34          7             0                  0
3        9           29          7             2                  2
4       21           37          9             3                  3
5       22           46         14             1                  1
6       17           34         16             0                  0
7        6           32         20             3                  3
8       25           39         22             1                  1
9       16           44         24             2                  2
10      19           46         25             1                  1
11      11           39         25             1                  1
12      29           35         25             2                  2
13      15           48         26             4

In [3]:
print(type(partition))

<class 'dict'>


In [14]:
print("Generated multilayer network with", multinet.number_of_edges(), "edges", multinet.number_of_nodes(), "nodes")

Generated multilayer network with 170 edges 100 nodes


In [15]:
import networkx as nx
import numpy as np
import pandas as pd

def save_graph_csv_poisson_int_ts(
    G: nx.Graph,
    out_csv: str,
    lam: float = 10.0,
    comm_attr: str = "mesoset",
    seed: int | None = 42,
    base_time: int = 0,
    sort_by_time: bool = True,
    make_strictly_increasing: bool = False,
    layer_index: int = 1,
    enforce_same_layer: bool = True,
):
    rng = np.random.default_rng(seed)

    edges = []
    if G.is_multigraph():
        for u, v, k in G.edges(keys=True):
            edges.append((u, v))
    else:
        for u, v in G.edges():
            edges.append((u, v))

    df_cols = ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    m = len(edges)
    if m == 0:
        pd.DataFrame(columns=df_cols).to_csv(out_csv, index=False)
        return

    edges_by_layer: dict[int, list[tuple]] = {}
    for (u, v) in edges:
        lu = u[layer_index] if isinstance(u, tuple) and len(u) > layer_index else None
        lv = v[layer_index] if isinstance(v, tuple) and len(v) > layer_index else None

        if enforce_same_layer and (lu != lv):
            raise ValueError(f"Edge endpoints are in different layers: {u} vs {v}")

        layer = lu
        edges_by_layer.setdefault(layer, []).append((u, v))


    shuffled_edges = []
    for layer in sorted(edges_by_layer.keys(), key=lambda x: (x is None, x)):
        layer_edges = edges_by_layer[layer]
        rng.shuffle(layer_edges)
        shuffled_edges.extend(layer_edges)

    ts = rng.poisson(lam=lam, size=len(shuffled_edges)).astype(np.int64)

    if make_strictly_increasing:
        ts = np.cumsum(ts + 1).astype(np.int64)

    ts = (ts + np.int64(base_time)).astype(np.int64)

    rows = []
    for i, (u, v) in enumerate(shuffled_edges):
        rows.append({
            "source": u[0] if isinstance(u, tuple) else u,
            "destination": v[0] if isinstance(v, tuple) else v,
            "timestamp": int(ts[i]),
            "source_commu": G.nodes[u].get(comm_attr, None),
            "destination_commu": G.nodes[v].get(comm_attr, None),
        })

    df = pd.DataFrame(rows)
    df["timestamp"] = df["timestamp"].astype("int64")

    if sort_by_time:
        df = df.sort_values(["timestamp"], kind="stable").reset_index(drop=True)

    df.to_csv(out_csv, index=False)

save_graph_csv_poisson_int_ts(
    multinet,
    out_csv="syn_net_test.csv",
    lam=5.0,
    comm_attr="mesoset",
    seed=42,
    base_time=0,
    sort_by_time=True,
    make_strictly_increasing=True,
)

In [1]:
from GraphSynthesiser import GraphSynthesiser

synth = GraphSynthesiser()

n_nodes = 50
n_layers = 20
p_list = [0.8] # prob nodes keep commu label across layers
theta = 0.2
n_sets = 5
mu_list = [0.2] # mu = 0 -> no inter-community links

for p in p_list:
    for mu in mu_list:
        for i in range(1,2):
            synth.synthesize_to_csv(
                f"syn_data/syn_net_p{p}_mu{mu}_l20.csv",
                n_nodes=n_nodes,
                n_layers=n_layers,
                p=p,
                theta=theta,
                n_sets=n_sets,
                mu=mu,
                k_min=2,
                k_max=10,
                t_k=-2,
                lam=5.0,
                seed=42,
                base_time=0,
                sort_by_time=True,
                make_strictly_increasing=True,
            )

/Users/acw721/Desktop/research/linkstream/.venv/lib/python3.12/site-packages/nxmultilayer.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
